In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

In [2]:
def clean_feature_names(df):
    # Function to clean feature names
    def clean_name(name):
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)
    df.columns = [clean_name(col) for col in df.columns]
    return df

In [3]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.9)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df

In [4]:
#Monomer composition
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp.csv')
df_mc_train = clean_feature_names(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(5568, 385)
(5568,)
(1392, 385)
(1392,)
0.540848821976949
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003319 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 574
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 61
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003156 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 591
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 65
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003340 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 4454, n

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3245,0.4078,0.5697,0.4788,0.7072,0.6771,0.2919,0.3879,0.5403,0.5408,0.7410,0.7110
LGBMRegressor,0.3025,0.4013,0.5500,0.5142,0.7184,0.6933,0.3077,0.4068,0.5547,0.5161,0.7198,0.7008
XGBRegressor,0.2663,0.3774,0.5161,0.5722,0.7565,0.7211,0.2575,0.3723,0.5075,0.5949,0.7723,0.7409
DecisionTreeRegressor,0.4260,0.4596,0.6526,0.3159,0.6388,0.6098,0.3300,0.4069,0.5745,0.4809,0.7100,0.6851
RandomForestRegressor,0.2918,0.3909,0.5402,0.5313,0.7309,0.6940,0.2809,0.3830,0.5300,0.5582,0.7477,0.7172
GradientBoostingRegressor,0.3363,0.4326,0.5799,0.4599,0.6893,0.6537,0.3396,0.4361,0.5827,0.4659,0.6945,0.6699
AdaBoostRegressor,0.5352,0.5878,0.7316,0.1404,0.4881,0.4413,0.5084,0.5783,0.7130,0.2004,0.5484,0.4956
SVR,0.3193,0.4006,0.5651,0.4871,0.7020,0.6890,0.3182,0.3984,0.5641,0.4995,0.7104,0.6991
LinearRegression,0.3832,0.4459,0.6191,0.3845,0.6272,0.6382,0.3678,0.4431,0.6065,0.4215,0.6515,0.6663
KNeighborsRegressor,0.3688,0.4452,0.6073,0.4077,0.6533,0.6170,0.3459,0.4280,0.5881,0.4560,0.6827,0.6521


In [5]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.731, -6.8256000000000006, -5.5903999999999...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.955200000000002, -6.548100000000004, -6.6...","[-6.933879999999999, -6.6374400000000024, -6.6...","[0.01606329978553693, 0.10749302488998852, 0.0..."
1,LGBMRegressor,"[-6.526702272819081, -6.066009374593228, -5.20...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.727196181481439, -6.397208501935518, -6.7...","[-6.994119502215978, -6.463343987145306, -6.67...","[0.1883182280973119, 0.07741237985911412, 0.23..."
2,XGBRegressor,"[-6.1672406, -6.3855033, -5.638817, -5.4592648...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.6936464, -6.1162605, -6.9831715, -5.33127...","[-6.4889793, -6.2697496, -6.8392115, -5.590609...","[0.18757308, 0.14719953, 0.19918199, 0.2519636..."
3,DecisionTreeRegressor,"[-7.0, -5.3, -5.66, -6.055, -4.66, -4.77, -4.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -7.0, -7.0, -7.0, -6.24, -6.055, -4.77...","[-7.0, -6.384, -6.539, -6.476000000000001, -6....","[0.0, 0.9014122253442096, 0.48617280878304997,..."
4,RandomForestRegressor,"[-6.395100000000003, -6.384899999999999, -5.51...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.76755, -6.445292857142862, -6.16430000000...","[-6.725228457199334, -6.518219695294574, -6.26...","[0.14158140147415038, 0.05748664598539583, 0.0..."
5,GradientBoostingRegressor,"[-5.668567342055251, -6.039859496903588, -5.17...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.854735841787733, -5.690708302068838, -5.9...","[-5.921176468949769, -5.739190040237821, -5.80...","[0.044170683802385924, 0.04009403991996833, 0...."
6,AdaBoostRegressor,"[-5.74875625455004, -5.875813953488371, -5.741...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.912108185991423, -5.749490182145266, -5.7...","[-6.000725270877288, -5.9537461866592425, -5.9...","[0.053628751311983254, 0.10350778954601611, 0...."
7,SVR,"[-4.73615154657683, -6.992846165732822, -4.614...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.856423794499231, -4.687987765328295, -5.2...","[-6.835100713090576, -4.7232137739723346, -5.2...","[0.029098626031245817, 0.03759478627232048, 0...."
8,LinearRegression,"[-5.022590521769931, -5.84668271405969, -4.924...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.7638686384894635, -4.889656937052838, -5....","[-5.867747558668874, -4.9277772487079385, -5.3...","[0.08609306998619239, 0.04851798752433319, 0.0..."
9,KNeighborsRegressor,"[-5.3566666666666665, -7.0, -5.853333333333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.433333333333334, -6.04, -5.38333333...","[-6.9093333333333335, -5.915333333333334, -6.2...","[0.04533333333333331, 0.3732178392782902, 0.23..."


In [6]:
result_df.to_csv('Results/Monomeric/Monomer_comp_results.csv')
prediction_df.to_csv('Results/Monomeric/Monomer_comp_prediction_data.csv')

In [7]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [8]:
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp.csv')
df_mc_train = clean_feature_names(df_mc_train)
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print(X_test)
print(y_test)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(5568, 243)
(5568,)
(1392, 243)
(1392,)
             A        dA       meA     Me_dA  Ala_indol_2_yl_  \
0    -0.501813 -0.349957  1.366587 -0.476635        -0.032844   
1    -0.501813 -0.349957  1.366587  0.793660        -0.032844   
2    -0.501813 -0.349957  1.366587 -0.476635        -0.032844   
3     0.624153 -0.349957 -0.555249 -0.476635        -0.032844   
4    -0.501813 -0.349957  1.366587  0.793660        -0.032844   
...        ...       ...       ...       ...              ...   
1387  4.752694 -0.349957 -0.555249 -0.476635        -0.032844   
1388  4.752694 -0.349957 -0.555249 -0.476635        -0.032844   
1389  3.439067 -0.349957 -0.555249 -0.476635        -0.032844   
1390 -0.501813 -0.349957 -0.555249 -0.476635        -0.032844   
1391  4.752694 -0.349957 -0.555249 -0.476635        -0.032844   

      dAla_indol_2_yl_  Ala_5_Tet_       Abu      dAbu    Me_Abu  ...  \
0            -0.013403   -0.023154 -0.122013 -0.107696 -0.052596  ...   
1            -0.013403   -0.02315

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3248,0.4078,0.5699,0.4783,0.7068,0.6767,0.2924,0.3877,0.5407,0.5401,0.7406,0.7111
LGBMRegressor,0.3025,0.4013,0.5500,0.5142,0.7184,0.6933,0.3077,0.4068,0.5547,0.5161,0.7198,0.7008
XGBRegressor,0.2663,0.3774,0.5161,0.5722,0.7565,0.7211,0.2575,0.3723,0.5075,0.5949,0.7723,0.7409
DecisionTreeRegressor,0.4255,0.4596,0.6523,0.3165,0.6386,0.6079,0.3271,0.4053,0.5719,0.4855,0.7127,0.6853
RandomForestRegressor,0.2920,0.3913,0.5404,0.5310,0.7307,0.6940,0.2808,0.3831,0.5299,0.5583,0.7477,0.7174
GradientBoostingRegressor,0.3363,0.4326,0.5799,0.4599,0.6893,0.6536,0.3396,0.4361,0.5827,0.4659,0.6945,0.6700
AdaBoostRegressor,0.5165,0.5741,0.7187,0.1704,0.4954,0.4428,0.4917,0.5650,0.7012,0.2267,0.5504,0.4987
SVR,0.3193,0.4006,0.5651,0.4871,0.7020,0.6890,0.3182,0.3984,0.5641,0.4995,0.7104,0.6991
LinearRegression,0.3832,0.4459,0.6191,0.3845,0.6272,0.6382,0.3678,0.4431,0.6065,0.4215,0.6515,0.6663
KNeighborsRegressor,0.3696,0.4459,0.6079,0.4064,0.6526,0.6163,0.3460,0.4283,0.5882,0.4557,0.6826,0.6519


In [9]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.7639, -6.788999999999999, -5.5768, -5.3622...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9807, -6.460200000000004, -6.6879, -6.330...","[-6.9362200000000005, -6.621940000000002, -6.6...","[0.025230727298276003, 0.13453448033868323, 0...."
1,LGBMRegressor,"[-6.526702272819081, -6.066009374593228, -5.20...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.727196181481439, -6.397208501935518, -6.7...","[-6.994119502215978, -6.463343987145306, -6.67...","[0.1883182280973119, 0.07741237985911412, 0.23..."
2,XGBRegressor,"[-6.1672406, -6.3855033, -5.638817, -5.4592648...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.6936464, -6.1162605, -6.9831715, -5.33127...","[-6.4889793, -6.2697496, -6.8392115, -5.590609...","[0.18757308, 0.14719953, 0.19918199, 0.2519636..."
3,DecisionTreeRegressor,"[-6.85, -5.3, -5.66, -6.0, -4.66, -4.68, -4.6,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.85, -5.47, -7.0, -6.24, -6.0, -4.6,...","[-7.0, -6.368, -6.381, -6.476000000000001, -6....","[0.0, 0.5564674294152353, 0.6674608602757169, ..."
4,RandomForestRegressor,"[-6.393755000000003, -6.3862999999999985, -5.4...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.7622, -6.372192857142861, -6.161300000000...","[-6.721050692307694, -6.485145028627907, -6.27...","[0.15504215752647438, 0.07677823169026261, 0.0..."
5,GradientBoostingRegressor,"[-5.66856734205525, -6.039859496903587, -5.174...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.854735841787731, -5.690708302068837, -5.9...","[-5.921176468949769, -5.739190040237821, -5.80...","[0.044170683802386895, 0.04009403991996849, 0...."
6,AdaBoostRegressor,"[-5.748756254550048, -5.875813953488374, -5.74...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.9121081859914275, -5.840991735537192, -5....","[-5.990851656612746, -5.976628366521899, -5.92...","[0.05147965959413099, 0.07549401119807704, 0.0..."
7,SVR,"[-4.735904656128343, -6.992520620266404, -4.61...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.856039216133023, -4.687835282485992, -5.2...","[-6.835031167503355, -4.723158848291465, -5.20...","[0.02901864100721433, 0.037719424608168406, 0...."
8,LinearRegression,"[-5.022590521769931, -5.846682714059691, -4.92...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.7638686384894635, -4.889656937052838, -5....","[-5.867747558668873, -4.927777248707938, -5.37...","[0.08609306998619208, 0.048517987524333304, 0...."
9,KNeighborsRegressor,"[-5.3566666666666665, -7.0, -5.853333333333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.433333333333334, -6.04, -5.38333333...","[-6.9093333333333335, -5.915333333333334, -6.2...","[0.04533333333333331, 0.3732178392782902, 0.23..."


In [10]:
const_col

['Ala_tBu_',
 'Me_Ala_indol_2_yl_',
 'Me_Abu_morpholino_',
 'meD',
 'Asp_Ph_2_NH2__',
 'Glu_3R_Me_',
 'Phe_CHF2_',
 'Me_Phe_4_Cl_',
 'Bn_4_OH__Gly',
 'Bu_Gly',
 'EtOEt_Gly',
 'PhEt_Gly',
 'isoamyl_Gly',
 '2_pyridylmethyl_Gly',
 'Me_Hph',
 'Hph_2_Cl_',
 'Hph_3_Cl_',
 'Hph_4_Cl_',
 'Hse_Et_',
 'Hyp_Et_',
 'dK',
 'meK',
 'Me_dK',
 'Lys_Cbz_',
 'Lys_iPr_',
 'Lys_Me_',
 'Me_Lys_Me_',
 'dLeu_3R_OH_',
 'dN',
 'Nle_CHF2_',
 'Nle_OH_',
 'Orn',
 '4Pal',
 'dPip',
 'Gln_Mes_',
 'Ser_Bn_',
 'Ser_EtNMe2_',
 'Ser_EtOH_',
 'Ser_isoamyl_',
 'dSer_Me_',
 'Ser_Ph_2_Cl__',
 'Ser_Ph_3_Cl__',
 'Ser_Pr_',
 'Me_Ser_isoamyl_',
 'Me_Ser_Pr_',
 'dT',
 'Me_Tza',
 '_N__O_Val',
 'meW',
 'Me_dW',
 'Trp_6_Br_',
 'Tyr_CHF2_',
 'dTyr_bR_OMe_',
 '_N__O_Tyr',
 'Mono3',
 'Mono4',
 'Mono5',
 'Mono15',
 'Mono17',
 'Mono18',
 'Mono19',
 'Mono20',
 'Mono23',
 'Mono24',
 'Mono25',
 'Mono32',
 'Mono33',
 'Mono36',
 'Mono48',
 'Mono49',
 'Mono50',
 'Mono51',
 'Mono52',
 'Mono53',
 'Mono54',
 'Mono55',
 'Mono56',
 'Mono57',
 'Mon

In [11]:
result_df.to_csv('Results/Monomeric/Monomer_comp_constRemoval_results.csv')
prediction_df.to_csv('Results/Monomeric/Monomer_comp_constRemoval_prediction_data.csv')

In [12]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    low_variance_columns = variances[variances < threshold].index.tolist()
    df_cleaned = df.drop(columns=low_variance_columns)
    return df_cleaned, low_variance_columns

In [13]:
df_train = pd.read_csv('features/Monomeric/Train_mon_comp.csv')
df_mc_train = clean_feature_names(df_train)
df_mc_train = df_mc_train.drop(['ID','SMILES','Permeability'],axis=1)
df_mc, const_col = remove_low_variance_columns(df_mc_train)
X_train = df_mc
y_train = df_train['Permeability']
print(X_train.shape)
print(y_train.shape)

df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(5568, 8)
(5568,)
(1392, 8)
(1392,)
0.3377384381344529
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026053 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 159
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 8
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000322 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 155
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 8
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000373 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 157
[LightGBM] [Info] Number of data points in the train set: 4454, number

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.3026284343110641


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.4268,0.4770,0.6533,0.3145,0.5741,0.5517,0.4210,0.4751,0.6489,0.3377,0.5863,0.5857
LGBMRegressor,0.4158,0.4766,0.6448,0.3322,0.5763,0.5537,0.4253,0.4818,0.6521,0.3310,0.5757,0.5759
XGBRegressor,0.4209,0.4734,0.6488,0.3240,0.5765,0.5544,0.4196,0.4752,0.6478,0.3400,0.5858,0.5900
DecisionTreeRegressor,0.4501,0.4847,0.6709,0.2771,0.5518,0.5372,0.4300,0.4788,0.6557,0.3236,0.5775,0.5800
RandomForestRegressor,0.4189,0.4746,0.6472,0.3272,0.5799,0.5515,0.4194,0.4752,0.6476,0.3404,0.5866,0.5840
GradientBoostingRegressor,0.4359,0.4906,0.6602,0.2998,0.5498,0.5415,0.4362,0.4879,0.6605,0.3138,0.5657,0.5634
AdaBoostRegressor,0.5464,0.5849,0.7392,0.1224,0.4265,0.4175,0.5403,0.5844,0.7350,0.1502,0.4542,0.4550
SVR,0.4624,0.4798,0.6800,0.2573,0.5274,0.5359,0.4637,0.4842,0.6810,0.2706,0.5385,0.5539
LinearRegression,0.5263,0.5342,0.7255,0.1547,0.3933,0.4644,0.5498,0.5437,0.7415,0.1352,0.3691,0.4529
KNeighborsRegressor,0.4935,0.5134,0.7025,0.2073,0.5189,0.4993,0.4789,0.4992,0.6920,0.2468,0.5288,0.5368


In [14]:
result_df.to_csv('Results/Monomeric/Monomer_comp_LVR_results.csv')
prediction_df.to_csv('Results/Monomeric/Monomer_comp_LVR_prediction_data.csv')

In [15]:
#AA composition
df_aac_train = pd.read_csv('features/Monomeric/Train_aac.csv')
X_train = df_aac_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_aac_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_aac_test = pd.read_csv('features/Monomeric/Test_aac.csv')
X_test = df_aac_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_aac_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
aac_comp,prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
aac_comp

(5568, 21)
(5568,)
(1392, 21)
(1392,)
0.3595706193257885
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001717 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 14
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001351 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 269
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 14
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001288 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 268
[LightGBM] [Info] Number of data points in the train set: 4454, nu

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.3183359872760805


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.3929,0.4658,0.6268,0.3689,0.6090,0.5223,0.4072,0.4687,0.6381,0.3596,0.6018,0.5444
LGBMRegressor,0.3974,0.4740,0.6304,0.3617,0.6014,0.5213,0.4070,0.4724,0.6379,0.3599,0.5999,0.5449
XGBRegressor,0.4005,0.4710,0.6328,0.3568,0.5995,0.5157,0.4025,0.4654,0.6344,0.3669,0.6067,0.5457
DecisionTreeRegressor,0.4234,0.4791,0.6507,0.3199,0.5771,0.4987,0.4133,0.4712,0.6429,0.3499,0.5954,0.5345
RandomForestRegressor,0.3936,0.4684,0.6274,0.3677,0.6068,0.5245,0.4057,0.4685,0.6369,0.3619,0.6021,0.5453
GradientBoostingRegressor,0.4129,0.4894,0.6426,0.3368,0.5818,0.4969,0.4307,0.4937,0.6563,0.3226,0.5686,0.5149
AdaBoostRegressor,0.5476,0.5907,0.7400,0.1205,0.4302,0.3765,0.5353,0.5874,0.7316,0.1581,0.4511,0.4321
SVR,0.4212,0.4725,0.6490,0.3235,0.5808,0.5044,0.4384,0.4715,0.6621,0.3105,0.5701,0.5241
LinearRegression,0.4972,0.5382,0.7051,0.2014,0.4490,0.4136,0.4934,0.5325,0.7025,0.2239,0.4735,0.4419
KNeighborsRegressor,0.5494,0.5564,0.7412,0.1175,0.4987,0.4113,0.5455,0.5475,0.7386,0.1419,0.5095,0.4602


In [16]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.500550000000003, -6.826199999999995, -6.27...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.913249999999997, -6.240000000000006, -6.8...","[-6.881735999999998, -6.373740000000005, -6.78...","[0.028017144465486973, 0.26747999999999694, 0...."
1,LGBMRegressor,"[-6.638170595687698, -6.51643399892424, -6.006...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.7417376372838875, -6.66712631547333, -6.3...","[-6.841209020230582, -6.615682756424268, -6.39...","[0.11925822816588465, 0.12014128449030413, 0.1..."
2,XGBRegressor,"[-6.6654763, -6.3752136, -6.1579, -5.4310584, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.680387, -6.3547006, -7.0676, -6.543622, -...","[-6.5275984, -6.462632, -6.737625, -6.3094015,...","[0.15065682, 0.17043608, 0.18810174, 0.2443106..."
3,DecisionTreeRegressor,"[-6.24, -6.89, -5.88, -5.92, -5.88, -4.6850000...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -7.0, -7.0, -7.0, -6.7866666666...","[-6.984, -6.37, -6.622, -7.0, -6.8480000000000...","[0.03200000000000003, 0.2599999999999998, 0.75..."
4,RandomForestRegressor,"[-6.598420000000002, -6.618756666666666, -6.17...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.731083333333336, -6.461633333333335, -6.6...","[-6.673107095238095, -6.461081333333335, -6.46...","[0.03637125733460169, 0.0969625237707824, 0.13..."
5,GradientBoostingRegressor,"[-5.676918214985284, -5.935319561156397, -5.75...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.676918214985284, -6.135903023729614, -5.7...","[-5.640045306603412, -5.980111488679471, -5.71...","[0.044944996948699224, 0.11232493146612496, 0...."
6,AdaBoostRegressor,"[-5.684627917217086, -5.586287958115178, -5.52...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.684627917217086, -5.684627917217086, -5.6...","[-5.6801698117801145, -5.6801698117801145, -5....","[0.012372009945787006, 0.012372009945787006, 0..."
7,SVR,"[-5.907995589209372, -5.80489270096378, -4.952...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.993748378907463, -5.6966988368754095, -6....","[-6.851887935677986, -5.7071287485411775, -6.0...","[0.07767860588792033, 0.0724807969188835, 0.14..."
8,LinearRegression,"[-4.71701123838122, -4.9840005058331585, -5.08...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-4.950330160527562, -4.815773589882825, -5.1...","[-4.982395294763519, -4.85237944539576, -5.090...","[0.031285344761025964, 0.03398560045812273, 0...."
9,KNeighborsRegressor,"[-5.633333333333333, -6.936666666666667, -4.53...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.973333333333334, -5.3500000000000005, -7....","[-6.8933333333333335, -5.826666666666666, -6.0...","[0.04173993557999636, 0.29902805516837017, 0.6..."


In [17]:
aac_comp.to_csv('Results/Monomeric/AAC_comp_results.csv')
prediction_df.to_csv('Results/Monomeric/AAC_comp_prediction_data.csv')

In [18]:
#Constant column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac.csv')
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

(5568, 21)
(5568,)
(1392, 21)
(1392,)


In [19]:
#LVR column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac.csv')
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
X_train, const_col = remove_low_variance_columns(X_train)

y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_mc = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_mc, X_train,y_train, X_test,  y_test)
result_df

(5568, 5)
(5568,)
(1392, 5)
(1392,)
0.322839164559773
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001206 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 148
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 5
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000270 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 149
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 5
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000377 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 149
[LightGBM] [Info] Number of data points in the train set: 4454, number 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.29074795722386804


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.4343,0.4856,0.6590,0.3024,0.5563,0.4867,0.4305,0.4795,0.6561,0.3228,0.5727,0.5298
LGBMRegressor,0.4251,0.4895,0.6520,0.3172,0.5632,0.4915,0.4207,0.4815,0.6486,0.3382,0.5819,0.5273
XGBRegressor,0.4334,0.4855,0.6583,0.3039,0.5573,0.4843,0.4242,0.4761,0.6513,0.3327,0.5799,0.5294
DecisionTreeRegressor,0.4573,0.4920,0.6763,0.2655,0.5350,0.4739,0.4454,0.4850,0.6674,0.2994,0.5570,0.5168
RandomForestRegressor,0.4246,0.4836,0.6516,0.3180,0.5660,0.4877,0.4251,0.4785,0.6520,0.3314,0.5774,0.5330
GradientBoostingRegressor,0.4439,0.5048,0.6662,0.2871,0.5368,0.4684,0.4407,0.5006,0.6638,0.3068,0.5565,0.5041
AdaBoostRegressor,0.5300,0.5829,0.7280,0.1487,0.4531,0.4088,0.5243,0.5828,0.7241,0.1754,0.4756,0.4514
SVR,0.4798,0.5001,0.6927,0.2294,0.4999,0.4493,0.4643,0.4919,0.6814,0.2697,0.5392,0.4869
LinearRegression,0.5534,0.5748,0.7439,0.1112,0.3335,0.3321,0.5589,0.5757,0.7476,0.1209,0.3483,0.3508
KNeighborsRegressor,0.6444,0.5948,0.8027,-0.0349,0.4124,0.3531,0.6025,0.5786,0.7762,0.0524,0.4438,0.3729


In [20]:
result_df.to_csv('Results/Monomeric/AAC_comp_LVR_results.csv')
prediction_df.to_csv('Results/Monomeric/AAC_comp_LVR_prediction_data.csv')

In [21]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.268815909090905, -6.362449999999999, -5.90...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.566785714285711, -6.620000000000004, -5.6...","[-5.970731626984126, -6.620000000000003, -5.86...","[0.28380019189363753, 0.2403331021727948, 0.14..."
1,LGBMRegressor,"[-5.948311090401905, -5.973137860089263, -5.96...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.936695763814133, -6.2310627719100475, -6....","[-5.871622266124943, -6.025779215053423, -5.93...","[0.12237459217754658, 0.1750083210433686, 0.17..."
2,XGBRegressor,"[-6.4977417, -5.594059, -6.3568273, -5.658737,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.2155223, -6.662711, -6.1895, -6.1758485, ...","[-6.570761, -6.625376, -6.6664224, -6.1857505,...","[0.33856457, 0.25292394, 0.30028653, 0.2128494..."
3,DecisionTreeRegressor,"[-5.835, -4.68, -6.62, -5.92, -5.88, -4.636666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.62, -5.65, -6.126666666666666, -7.0...","[-6.992, -6.62, -6.04, -6.126666666666667, -7....","[0.016000000000000014, 0.24033310217279674, 0...."
4,RandomForestRegressor,"[-5.935308787878788, -5.556749999999998, -6.02...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.986132002164502, -6.554120000000001, -6.0...","[-6.1452391242646245, -6.464657166666669, -6.0...","[0.27471348057261297, 0.17789047807694552, 0.2..."
5,GradientBoostingRegressor,"[-6.2374911588428175, -5.2569626399384415, -5....",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.2374911588428175, -5.191881768314697, -6....","[-5.8204542019099135, -5.2668113543421455, -5....","[0.3779850262041275, 0.17423978358973458, 0.39..."
6,AdaBoostRegressor,"[-5.577162859248355, -5.577162859248355, -5.57...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.577162859248355, -5.327202072538858, -5.5...","[-5.530393237290519, -5.445401068041381, -5.53...","[0.04767788858740372, 0.10245599330061501, 0.0..."
7,SVR,"[-5.030002730425981, -5.12771559706299, -5.049...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-4.906274728874786, -4.667325217405786, -4.8...","[-4.944220770106485, -4.60812248543305, -4.833...","[0.06036269307082572, 0.035312677775686314, 0...."
8,LinearRegression,"[-5.202706200677448, -5.337151471749982, -5.09...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.215524140464273, -5.351833687978308, -5.2...","[-5.220891888894108, -5.361749492002933, -5.23...","[0.013582612809786643, 0.012403271791710595, 0..."
9,KNeighborsRegressor,"[-5.706666666666667, -6.28, -4.71, -5.24666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-4.95, -5.926666666666667, -5.37666666666666...","[-4.913333333333334, -5.683999999999999, -5.16...","[0.261057677747871, 0.33822017812201727, 0.329..."


In [22]:
const_col

['I',
 'Q',
 'S',
 'D',
 'R',
 'W',
 'E',
 'T',
 'X',
 'V',
 'H',
 'N',
 'C',
 'M',
 'Y',
 'K']

In [5]:
from sklearn.model_selection import GridSearchCV
def train_and_test_predict_with_tuning(models, param_grids, X_train, y_train, X_test, y_test):
   
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []
        test_predictions_folds = []

        best_params = None

        # hyperparameter tuning 
        if model_name in param_grids and param_grids[model_name]:
            default_params = model.get_params()
            print(model_name, ': Default params', default_params)
            grid_search = GridSearchCV(
                estimator=model, 
                param_grid=param_grids[model_name], 
                cv=kf,
                scoring='neg_mean_squared_error', 
                n_jobs=-1)
            grid_search.fit(X_train, y_train)
            model = grid_search.best_estimator_
            best_params = grid_search.best_params_
            print(model_name)
            print(": best params",best_params)
        else:
            default_params = model.get_params()
            print(model_name, ': Default params', default_params)
            best_params = {}
            print(model_name, ':Used Default params')

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)  
            test_predictions_folds.append(predictions_test_fold)

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,
            'Best Parameters': best_params
        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df


In [6]:
param_grids = {
        'ExtraTreesRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'max_depth': [None,1,5, 10, 20],
            'min_samples_split': [2, 5, 10]
        },
        'LGBMRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.05, 0.1],
            'num_leaves': [31, 50, 100]
        },
        'DecisionTreeRegressor': {
            'max_depth': [None, 10, 20, 50, 100],
            'min_samples_split': [2, 5, 10]
        },
        'RandomForestRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'max_depth': [None, 1, 5, 10, 20],
            'min_samples_split': [2, 5, 10]
        },
        'GradientBoostingRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7, 10]
        },
        'AdaBoostRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.1, 1.0]
        },
        'SVR': {
            'C': [0.001, 0.1, 1, 10],
            'epsilon': [0.1, 0.2, 0.5],
            'gamma': [0.001, 0.1, 1, 10]
        },
        'KNeighborsRegressor': {
            'n_neighbors': [3, 5, 10],
            'weights': ['uniform', 'distance']
        },
        'MLPRegressor': {
            'hidden_layer_sizes': [(50,), (100,), (50, 50)],
            'learning_rate': ['constant', 'adaptive'],
            'max_iter': [100,200, 400]
}
    }


In [7]:
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp.csv')
df_mc_train = clean_feature_names(df_mc_train)
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
print(X_test)
print(y_test)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict_with_tuning(models,param_grids, X_train,y_train, X_test,  y_test)
result_df

(5568, 243)
(5568,)
(1392, 243)
(1392,)
             A        dA       meA     Me_dA  Ala_indol_2_yl_  \
0    -0.501813 -0.349957  1.366587 -0.476635        -0.032844   
1    -0.501813 -0.349957  1.366587  0.793660        -0.032844   
2    -0.501813 -0.349957  1.366587 -0.476635        -0.032844   
3     0.624153 -0.349957 -0.555249 -0.476635        -0.032844   
4    -0.501813 -0.349957  1.366587  0.793660        -0.032844   
...        ...       ...       ...       ...              ...   
1387  4.752694 -0.349957 -0.555249 -0.476635        -0.032844   
1388  4.752694 -0.349957 -0.555249 -0.476635        -0.032844   
1389  3.439067 -0.349957 -0.555249 -0.476635        -0.032844   
1390 -0.501813 -0.349957 -0.555249 -0.476635        -0.032844   
1391  4.752694 -0.349957 -0.555249 -0.476635        -0.032844   

      dAla_indol_2_yl_  Ala_5_Tet_       Abu      dAbu    Me_Abu  ...  \
0            -0.013403   -0.023154 -0.122013 -0.107696 -0.052596  ...   
1            -0.013403   -0.02315

c:\Python312\Lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The LGBMRegressor or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004380 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 639
[LightGBM] [Info] Number of data points in the train set: 5568, number of used features: 71
[LightGBM] [Info] Start training from score -5.742906
LGBMRegressor
: best params {'learning_rate': 0.1, 'n_estimators': 400, 'num_leaves': 50}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001399 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 574
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 61
[LightGBM] [Info] Start training from score -5.738310
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000884 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[L

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.2794,0.3866,0.5286,0.5514,0.7426,0.7083,0.2726,0.3798,0.5221,0.5692,0.7548,0.7267
LGBMRegressor,0.2694,0.3732,0.5191,0.5674,0.7552,0.7209,0.2516,0.3656,0.5016,0.6025,0.7763,0.7456
XGBRegressor,0.2699,0.3799,0.5195,0.5666,0.7528,0.7135,0.2559,0.3704,0.5058,0.5957,0.7728,0.7435
DecisionTreeRegressor,0.3572,0.4278,0.5977,0.4264,0.6740,0.6369,0.2845,0.3842,0.5334,0.5504,0.7431,0.7150
RandomForestRegressor,0.2798,0.3845,0.5290,0.5507,0.7422,0.7077,0.2666,0.3769,0.5163,0.5788,0.7612,0.7328
GradientBoostingRegressor,0.2629,0.3712,0.5127,0.5779,0.7605,0.7257,0.2495,0.3645,0.4995,0.6057,0.7784,0.7465
AdaBoostRegressor,0.4825,0.5214,0.6946,0.2253,0.4749,0.4591,0.4732,0.5162,0.6879,0.2522,0.5047,0.5209
SVR,0.3118,0.3997,0.5584,0.4993,0.7084,0.6880,0.3147,0.3986,0.5609,0.5028,0.7107,0.6959
LinearRegression,0.3994,0.4479,0.6319,0.3588,0.6116,0.6371,0.3637,0.4399,0.6031,0.4253,0.6543,0.6680
KNeighborsRegressor,0.3548,0.4380,0.5957,0.4303,0.6612,0.6292,0.3452,0.4271,0.5875,0.4545,0.6775,0.6504


In [8]:
result_df.to_csv('Results/Monomeric/Monomer_comp_constRemoval_results_with_HPT.csv')
prediction_df.to_csv('Results/Monomeric/Monomer_comp_constRemoval_prediction_data_with_HPT.csv')